In [ ]:
# BLOCO 1: Instalação (necessária em cada nova sessão do Colab)
!pip install -q notion-client

In [ ]:
# BLOCO 2: Montagem do Drive e Configurações do Lote
from google.colab import drive, userdata
import json
import os
import sys

drive.mount('/content/drive')

# ----------------------------------------------------------------------
# PARÂMETROS DA PUBLICAÇÃO EM LOTE
# ----------------------------------------------------------------------
ID_MESA = "DIA"                # Mesa cadastrada em notion_config.json (ex.: "ID", "DiT", "DIA")
DEFAULT_DATE = "2026-03-23"    # Aplicada a TODAS as sessões publicadas (YYYY-MM-DD)
SLEEP_SECONDS = 0.3            # Pausa entre publicações (limite de requisições do Notion)
FORCE_UPDATE = False            # True para garantir a versão mais recente do repo
# ----------------------------------------------------------------------

# Mesma pasta do Bloco 3 do transcritor_rpg.ipynb: é de onde as crônicas são lidas
PROJECT_FOLDER = "/content/drive/MyDrive/rpg-transcription"

if not os.path.exists('rpg_transcription'):
    !git clone https://github.com/ThaisMosken/rpg_transcription.git
elif FORCE_UPDATE:
    !git -C rpg_transcription pull

sys.path.append('/content/rpg_transcription')

from utils.session_helper import build_folder_layout
layout = build_folder_layout(PROJECT_FOLDER, ID_MESA)

with open(os.path.join(layout["config_folder"], "notion_config.json"), "r", encoding="utf-8") as f:
    NOTION_CONFIG = json.load(f)

NOTION_TOKEN = userdata.get('NOTION_TOKEN')

print(f"✅ Ambiente pronto para publicar o backlog da mesa '{ID_MESA}'.")

In [ ]:
# BLOCO 3: Publicação em lote no Notion (sessões já existentes são puladas)
from src.notion_backlog import publish_backlog_sessions

publish_backlog_sessions(
    notion_token=NOTION_TOKEN,
    notion_config=NOTION_CONFIG,
    table_id=ID_MESA,
    chronicles_folder=layout["archive_chronicles_folder"],
    default_date=DEFAULT_DATE,
    sleep_seconds=SLEEP_SECONDS
)